# 02 · Boulder's Disappearing Childhood: Destiny or Policy?

### School-age (5–17) decline in the City of Boulder vs. Colorado, college-town & national peers — Hamilton–Perry to 2050

**Brian Keegan** · *Charting Boulder*, Boulder Reporting Lab · sequel to *"Boulder's next political divide is generational"* (Oct 2025)

**Stage 2 of 2 — modeling, analysis, visualization.** All acquisition, cleaning, and validation
happen in `01-data-retrieval.ipynb`, which writes `data/processed/`. This notebook reads that
contract and does no downloading. Provenance (which inputs are live vs. synthetic) is printed from
the manifest below; nothing here is a finding until the manifest shows the relevant inputs `live`.

**Claim discipline.** Descriptive only. H1 = is Boulder's decline *faster* than peers? H2 = does
steeper decline *travel with* restrictive land use? No causal effect of zoning is estimated. The
projection is a **no-major-shock baseline** rolling 2000–2020 dynamics forward; Boulder's 2024–25
reforms post-date the base period and are not in it.

## 1 · Setup & engine

Imports, deterministic seed, house chart style, the projection frame, and the Hamilton–Perry
engine (trended hybrid CCR/CCD + CWR). Engine functions are defined once here and reused by the
county rebuild (Module 3) and the place projector (Module 4).

In [1]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.options.display.max_columns = 100
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 20260611
RNG = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 5.2),
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11})
HIGHLIGHT, CONTEXT, ACCENT = "tab:red", "tab:gray", "tab:blue"

PROC = Path("data/processed"); OUT = Path("output"); OUT.mkdir(exist_ok=True)
BASE_YEARS = [1990, 2000, 2010, 2020]
PROJ_YEARS = [2030, 2040, 2050]
STEP, MAX_AGE = 10, 85
AGES = np.arange(0, MAX_AGE + 1)
SCHOOL_AGE = (AGES >= 5) & (AGES <= 17)
UNDER5 = AGES < 5
WOMEN_FERT = (AGES >= 15) & (AGES <= 49)
BANDS = {"0-4": UNDER5, "5-17": SCHOOL_AGE, "18-24": (AGES >= 18) & (AGES <= 24),
         "25-64": (AGES >= 25) & (AGES <= 64), "65+": AGES >= 65}
AGE5 = [(0,4),(5,9),(10,14),(15,19),(20,24),(25,29),(30,34),(35,39),(40,44),
        (45,49),(50,54),(55,59),(60,64),(65,69),(70,74),(75,79),(80,84),(85,200)]
def age5_label(a, b): return f"{a}-{b}" if b < 200 else "85+"
print("setup ok · project to", PROJ_YEARS[-1])

setup ok · project to 2050


In [2]:
def school_age_total(v): return float(np.asarray(v)[SCHOOL_AGE].sum())
def under5_total(v): return float(np.asarray(v)[UNDER5].sum())

def cohort_change_ratios(p0, p1, step=STEP):
    p0 = np.asarray(p0, float); p1 = np.asarray(p1, float)
    ccr = np.full(MAX_AGE + 1, np.nan)
    for a in range(step, MAX_AGE + 1):
        denom = p0[MAX_AGE - step:].sum() if a == MAX_AGE else p0[a - step]
        ccr[a] = p1[a] / denom if denom > 0 else np.nan
    return ccr

def cohort_change_differences(p0, p1, step=STEP):
    p0 = np.asarray(p0, float); p1 = np.asarray(p1, float)
    ccd = np.full(MAX_AGE + 1, np.nan)
    for a in range(step, MAX_AGE + 1):
        base = p0[MAX_AGE - step:].sum() if a == MAX_AGE else p0[a - step]
        ccd[a] = p1[a] - base
    return ccd

def child_woman_ratio(p, step=STEP):
    p = np.asarray(p, float); w = p[WOMEN_FERT].sum()
    return p[:step].sum() / w if w > 0 else np.nan

def fit_trend_extrapolate(values, n_future, damp=0.5, lo=0.2, hi=3.0):
    v = np.asarray(values, float); v = np.where(np.isfinite(v) & (v > 0), v, np.nan)
    if np.sum(np.isfinite(v)) < 2:
        last = np.nanmean(v) if np.isfinite(np.nanmean(v)) else 1.0
        return np.clip(np.full(n_future, last), lo, hi)
    idx = np.arange(len(v)); ok = np.isfinite(v)
    slope, intercept = np.polyfit(idx[ok], np.log(v[ok]), 1)
    last_log = np.log(v[ok][-1]); out = []
    for k in range(1, n_future + 1):
        trend_log = intercept + slope * (len(v) - 1 + k)
        out.append(np.exp(damp * trend_log + (1 - damp) * last_log))
    return np.clip(np.array(out), lo, hi)

def hp_project(age_by_year, base_years, n_steps, hybrid=True, trend_cwr=True, cwr_override=None):
    yrs = sorted(base_years); vecs = [np.asarray(age_by_year[y], float) for y in yrs]
    ccr_v = [cohort_change_ratios(vecs[i], vecs[i+1]) for i in range(len(vecs)-1)]
    ccd_v = [cohort_change_differences(vecs[i], vecs[i+1]) for i in range(len(vecs)-1)]
    cwr_v = [child_woman_ratio(v) for v in vecs]
    recent = vecs[-1] - np.concatenate([[np.nan]*STEP, vecs[-2][:-STEP]])
    use_ccd = hybrid & (recent > 0)
    ccr_f = np.vstack([fit_trend_extrapolate([cv[a] for cv in ccr_v], n_steps) for a in range(MAX_AGE+1)]).T
    ccd_f = np.vstack([fit_trend_extrapolate([cv[a]+5.0 for cv in ccd_v], n_steps, lo=-1e9, hi=1e9) for a in range(MAX_AGE+1)]).T - 5.0
    if trend_cwr and cwr_override is None:
        cwr_f = fit_trend_extrapolate(cwr_v, n_steps, lo=0.05, hi=1.5)
    else:
        cwr_f = np.full(n_steps, cwr_override if cwr_override is not None else np.nanmean(cwr_v))
    out = {}; cur = vecs[-1].copy(); last = yrs[-1]
    base_kids_shape = vecs[-1][:STEP] / vecs[-1][:STEP].sum() if vecs[-1][:STEP].sum() > 0 else np.ones(STEP)/STEP
    for s in range(n_steps):
        nxt = np.zeros(MAX_AGE + 1)
        for a in range(STEP, MAX_AGE + 1):
            src = cur[MAX_AGE - STEP:].sum() if a == MAX_AGE else cur[a - STEP]
            nxt[a] = max(src + ccd_f[s, a], 0.0) if use_ccd[a] else max(src * ccr_f[s, a], 0.0)
        nxt[:STEP] = max(cwr_f[s] * nxt[WOMEN_FERT].sum(), 0.0) * base_kids_shape
        out[last + (s+1)*STEP] = nxt; cur = nxt
    return out
print("engine ready")

engine ready


## 2 · Load the processed contract

Reads only `data/processed/`. The manifest banner shows each input's mode — **a number is a
finding only if its inputs are `live`.** Heavy validation already ran in notebook 01; here we keep
light contract checks (shape, FIPS width, age coverage) so a stale or partial contract fails fast.

In [3]:
man = pd.DataFrame(json.loads((PROC / "manifest.json").read_text()))
print("INPUT PROVENANCE (from 01-data-retrieval.ipynb):")
print(man[["name", "mode", "rows"]].to_string(index=False))
SYNTH = man[man["mode"].str.contains("synth")]["name"].tolist()
if SYNTH:
    print("\n!! synthetic inputs (NOT findings):", ", ".join(SYNTH))

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/manifest.json'

In [ ]:
# registry -> places_df (county_fips parsed from JSON list)
places_df = pd.read_csv(PROC / "places_registry.csv", dtype={"place_fips": str})
places_df["county_fips"] = places_df["county_fips_json"].map(json.loads)
places_df["place_fips"] = places_df["place_fips"].str.zfill(7)
ERIE_SPLIT = {"08013": 0.55, "08123": 0.45}
PLACE_FIPS = places_df["place_fips"].tolist()
ALL_COUNTIES = sorted({c for cs in places_df["county_fips"] for c in cs})

def long_to_age_dict(df, idcol):
    # {fips: {year: vec86}} from tidy long [idcol, year, age, pop]
    out = {}
    for f, g in df.groupby(idcol):
        out[f] = {}
        for y, gy in g.groupby("year"):
            v = np.zeros(MAX_AGE + 1)
            v[gy["age"].to_numpy()] = gy["pop"].to_numpy()
            out[f][int(y)] = v
    return out

place_age = long_to_age_dict(
    pd.read_csv(PROC / "place_age_syoa.csv", dtype={"place_fips": str}).assign(
        place_fips=lambda d: d.place_fips.str.zfill(7)), "place_fips")
county_age = long_to_age_dict(
    pd.read_csv(PROC / "county_age_syoa.csv", dtype={"county_fips": str}).assign(
        county_fips=lambda d: d.county_fips.str.zfill(5)), "county_fips")
irs_df = pd.read_csv(PROC / "irs_flows.csv", dtype={"origin_fips": str, "dest_fips": str})
irs_df["origin_fips"] = irs_df["origin_fips"].str.zfill(5); irs_df["dest_fips"] = irs_df["dest_fips"].str.zfill(5)
cov_df = pd.read_csv(PROC / "covariates.csv", dtype={"place_fips": str}).assign(
    place_fips=lambda d: d.place_fips.str.zfill(7))
hauer5 = pd.read_csv(PROC / "county_control_5yr.csv", dtype={"county_fips": str}).assign(
    county_fips=lambda d: d.county_fips.str.zfill(5))
hauer5_lookup = {(r.county_fips, int(r.year), r.age_group): r.pop for r in hauer5.itertuples()}
print("loaded:", len(place_age), "places,", len(county_age), "counties,",
      f"{len(irs_df):,} flows, {len(cov_df)} covariate rows")

In [ ]:
# light contract assertions (full validation lives in nb 01)
assert set(place_age) == set(PLACE_FIPS), "place_age vs registry mismatch"
for f in PLACE_FIPS:
    assert set(place_age[f]) == set(BASE_YEARS), f"{f} base years"
    assert all(place_age[f][y].shape == (MAX_AGE + 1,) for y in BASE_YEARS)
for c in ALL_COUNTIES:
    assert c in county_age, f"missing county history {c}"
assert set(cov_df["place_fips"]) == set(PLACE_FIPS)
assert irs_df["origin_fips"].str.len().eq(5).all()
boulder_fips = places_df.loc[places_df.name.str.startswith("Boulder"), "place_fips"].iloc[0]
print("OK contract · Boulder =", boulder_fips, "· counties", len(ALL_COUNTIES))

## 3 · Single-year county control (Hauer rebuild) + validation

Hauer's published control is five-year, so 5–17 cannot be summed cleanly. We **rebuild it at
single-year-of-age**: project each county's single-year history (1990→2020) with the HP engine,
then scale each five-year age band to the Hauer SSP2 control totals (projected years only; the 2020
base is left as observed). For Colorado counties the history is **real DOLA single-year**; non-CO
peers use NHGIS (synthetic until keyed). Validated out-of-sample on a decade-aligned 2000→2020
backtest. Because the control is itself HP-family, anchoring places to it is *consistency*, not
independent validation.

In [ ]:
def rebuild_county_singleyear(cfips):
    hist = county_age[cfips]
    proj = hp_project({y: hist[y] for y in BASE_YEARS}, BASE_YEARS, n_steps=len(PROJ_YEARS))
    chain = {2020: hist[2020].copy()}
    for y in PROJ_YEARS:
        v = proj[y].copy()
        for (a, b) in AGE5:                          # control projected years to Hauer 5yr totals
            lab = age5_label(a, b); tgt = hauer5_lookup.get((cfips, y, lab))
            if tgt is None: continue
            mask = (AGES >= a) & (AGES <= min(b, MAX_AGE))
            cur = v[mask].sum()
            if cur > 0: v[mask] *= tgt / cur
        chain[y] = v
    return chain

hauer_control = {c: rebuild_county_singleyear(c) for c in ALL_COUNTIES}
print("rebuilt single-year control for", len(hauer_control), "counties")

# rebuild backtest: 1990,2000 -> 2020 vs actual, by band
rows = []
for c in ALL_COUNTIES:
    hist = county_age[c]
    pred = hp_project({y: hist[y] for y in [1990, 2000]}, [1990, 2000], n_steps=2)[2020]
    act = hist[2020]
    for bnd, m in BANDS.items():
        a = act[m].sum(); p = pred[m].sum()
        rows.append((c, bnd, abs(p - a) / a * 100 if a > 0 else np.nan))
rebuild_bt = pd.DataFrame(rows, columns=["county", "band", "ape"])
print("rebuild backtest (2000->2020) median APE by band:")
print(rebuild_bt.groupby("band")["ape"].median().round(1).to_string())

In [ ]:
for c, chain in hauer_control.items():
    assert set(chain) == {2020, *PROJ_YEARS}
    for y, v in chain.items():
        assert v.shape == (MAX_AGE + 1,) and np.all(v >= 0)
assert rebuild_bt["ape"].notna().any()
print("OK · single-year control rebuilt + backtested, non-negative")

## 4 · Place-level Hamilton–Perry projection to 2050

Trended hybrid CCR/CCD + CWR per place, then **damped toward the county SSP2 trajectory** (λ).
Places don't tile counties, so we shrink each place's broad-band growth toward its county's growth
rather than strictly summing — a documented softening. Erie blends Boulder+Weld via `ERIE_SPLIT`.

In [ ]:
LAMBDA = 0.35
def county_band_growth(county_list, step_year, prev_year):
    if len(county_list) == 1:
        w = {county_list[0]: 1.0}
    else:
        w = {c: ERIE_SPLIT.get(c, 1.0 / len(county_list)) for c in county_list}
        s = sum(w.values()); w = {c: x / s for c, x in w.items()}
    g = {}
    for b, m in BANDS.items():
        num = den = 0.0
        for c, ww in w.items():
            num += ww * hauer_control[c][step_year][m].sum()
            den += ww * hauer_control[c][prev_year][m].sum()
        g[b] = num / den if den > 0 else 1.0
    return g

def damp_to_county(proj, prev_vec, county_list, step_year, prev_year, lam=LAMBDA):
    cg = county_band_growth(county_list, step_year, prev_year); out = proj.copy()
    for b, m in BANDS.items():
        idx = np.where(m)[0]; cur = proj[idx].sum(); prev = prev_vec[idx].sum()
        if prev <= 0 or cur <= 0: continue
        blended = (1 - lam) * (cur / prev) + lam * cg[b]
        out[idx] = proj[idx] * (prev * blended / cur)
    return out

place_proj = {}
for _, row in places_df.iterrows():
    fips = row["place_fips"]; base = place_age[fips]
    raw = hp_project({y: base[y] for y in BASE_YEARS}, BASE_YEARS, n_steps=len(PROJ_YEARS))
    chain = {2020: base[2020]}
    for i, y in enumerate(PROJ_YEARS):
        prev_y = 2020 if i == 0 else PROJ_YEARS[i - 1]
        chain[y] = damp_to_county(raw[y], chain[prev_y], row["county_fips"], y, prev_y)
    place_proj[fips] = chain
for fips, chain in place_proj.items():
    assert set(chain) == {2020, *PROJ_YEARS} and all(np.all(v >= 0) for v in chain.values())
print("OK · projected", len(place_proj), "places to 2050")

## 5 · Validation backtest → uncertainty envelope

Hold out 2020; build CCRs on 1990→2010; project to 2020; compare predicted vs. actual school-age.
Median absolute % error becomes the band drawn around the 2050 projection. HP is validated to ~15
years; 2050 is ~30 years out, so the envelope is a **floor** on uncertainty, not a CI.

In [ ]:
bt = []
for fips in place_proj:
    base = place_age[fips]
    pred = hp_project({y: base[y] for y in [1990, 2000, 2010]}, [1990, 2000, 2010], n_steps=1)[2020]
    aa = school_age_total(base[2020]); pa = school_age_total(pred)
    bt.append((fips, aa, pa, abs(pa - aa) / aa * 100 if aa > 0 else np.nan))
place_bt = pd.DataFrame(bt, columns=["place_fips", "actual_5_17", "pred_5_17", "ape"])
ENVELOPE = float(np.nanmedian(place_bt["ape"]))
print(f"place school-age backtest (1990-2010 -> 2020): median APE = {ENVELOPE:.1f}%")

## 6 · H1 — Is Boulder's school-age decline faster than peers?

Level *and* change reported together (rate-of-decline alone invites a floor effect). Boulder's
**percentile rank among college-town peers** — the within-type comparison the "destiny-for-a-type"
counterargument can't absorb — is the headline.

In [ ]:
def metrics_row(fips):
    ch = place_proj[fips]
    sa20, sa50 = school_age_total(ch[2020]), school_age_total(ch[2050])
    u20, u50 = under5_total(ch[2020]), under5_total(ch[2050])
    tot20 = ch[2020].sum()
    return dict(place_fips=fips, sa_2020=sa20, sa_2050=sa50,
                sa_pct_change=(sa50 - sa20) / sa20 * 100 if sa20 else np.nan,
                sa_share_2020=sa20 / tot20 * 100 if tot20 else np.nan,
                u5_pct_change=(u50 - u20) / u20 * 100 if u20 else np.nan)
h1 = pd.DataFrame([metrics_row(f) for f in place_proj]).merge(
    places_df[["place_fips", "name", "tier"]], on="place_fips")
b = h1[h1.place_fips == boulder_fips].iloc[0]
peers2 = h1[h1.tier == 2]
rank_change = float((-peers2.sa_pct_change < -b.sa_pct_change).mean() * 100)
rank_level = float((peers2.sa_share_2020 < b.sa_share_2020).mean() * 100)
print("H1:")
print(f"  Boulder 5-17 change 2020->2050 : {b.sa_pct_change:+.1f}%   (under-5 {b.u5_pct_change:+.1f}%)")
print(f"  steeper-decline pctile vs college towns : {rank_change:.0f}th")
print(f"  child-share level pctile (low=few kids) : {rank_level:.0f}th")
h1.sort_values("sa_pct_change")[["name", "tier", "sa_share_2020", "sa_pct_change", "u5_pct_change"]].head(8)

In [ ]:
fig, ax = plt.subplots()
yrs = [2020, *PROJ_YEARS]
for fips, ch in place_proj.items():
    idx = [school_age_total(ch[y]) / school_age_total(ch[2020]) * 100 for y in yrs]
    isb = fips == boulder_fips
    ax.plot(yrs, idx, color=HIGHLIGHT if isb else CONTEXT, lw=3 if isb else 1,
            alpha=1 if isb else 0.35, zorder=3 if isb else 1)
bidx = np.array([school_age_total(place_proj[boulder_fips][y]) / school_age_total(place_proj[boulder_fips][2020]) * 100 for y in yrs])
ax.fill_between(yrs, bidx*(1-ENVELOPE/100), bidx*(1+ENVELOPE/100), color=HIGHLIGHT, alpha=0.12, zorder=2)
ax.axhline(100, color="k", lw=0.6, ls=":")
ax.set_title("School-age (5–17) indexed to 2020 — Boulder (red) vs peers")
ax.set_ylabel("Index, 2020 = 100"); ax.set_xlabel("Year"); ax.set_xticks(yrs)
fig.tight_layout()

## 7 · Migration vs. fertility — the destiny test

(1) city-grain proxy splitting child change into fertility-implied vs. residual net migration;
(2) frozen-fertility counterfactual isolating the fertility-trend share; (3) the IRS flow exhibit
showing where Boulder County's out-migrants actually went (county grain; corroboration only).

In [ ]:
# 1) city-grain proxy
pr = []
for fips in place_proj:
    base = place_age[fips]
    implied = child_woman_ratio(base[2010]) * base[2020][WOMEN_FERT].sum()
    actual = base[2020][:STEP].sum()
    pr.append((fips, actual - implied, (actual - implied) / actual * 100 if actual else np.nan))
mig_proxy = pd.DataFrame(pr, columns=["place_fips", "net_child_mig_proxy", "mig_share_pct"]).merge(
    places_df[["place_fips", "name", "tier"]], on="place_fips")
print("city-grain migration proxy (negative = net child out-migration), most negative:")
print(mig_proxy.sort_values("mig_share_pct")[["name", "tier", "mig_share_pct"]].head(6).to_string(index=False))

In [ ]:
# 2) frozen-fertility counterfactual for every place
ff = []
for fips in place_proj:
    base = place_age[fips]
    flat = hp_project({y: base[y] for y in BASE_YEARS}, BASE_YEARS, n_steps=len(PROJ_YEARS),
                      trend_cwr=False, cwr_override=np.nanmean([child_woman_ratio(base[y]) for y in BASE_YEARS]))
    sa_main = school_age_total(place_proj[fips][2050]); sa_frozen = school_age_total(flat[2050])
    ff.append((fips, sa_main, sa_frozen, sa_frozen - sa_main))
ff = pd.DataFrame(ff, columns=["place_fips", "sa2050_main", "sa2050_frozen", "fertility_trend_gap"]).merge(
    places_df[["place_fips", "name"]], on="place_fips")
_bff = ff[ff.place_fips == boulder_fips].iloc[0]
print("frozen-fertility counterfactual (Boulder):")
print(f"  5-17 2050 trended CWR {_bff.sa2050_main:.0f} | frozen CWR {_bff.sa2050_frozen:.0f}"
      f" | fertility-trend contribution {_bff.fertility_trend_gap:+.0f}")
print("  (remainder of the change is survival+migration = the non-fertility share)")

In [ ]:
# 3) IRS flow exhibit: Boulder County (08013) recent outflow destinations
out = (irs_df[(irs_df.origin_fips == "08013") & (irs_df.year >= irs_df.year.max() - 5)]
       .groupby("dest_fips")["n_migrants"].sum().sort_values(ascending=False).head(8))
CHEAPER = {"08123", "08069", "08001", "08005", "08035"}  # Weld, Larimer, Adams, Arapahoe, Douglas
if len(out):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    cols = [HIGHLIGHT if d in CHEAPER else CONTEXT for d in out.index]
    ax.barh(range(len(out)), out.values, color=cols); ax.set_yticks(range(len(out)))
    ax.set_yticklabels(out.index); ax.invert_yaxis()
    ax.set_title("Where Boulder County out-migrants went (red = cheaper Front Range)")
    ax.set_xlabel(f"Migrants, {irs_df.year.max()-5}-{irs_df.year.max()} (IRS SOI)")
    fig.tight_layout()
else:
    print("no Boulder-origin flows in window")

## 8 · H2 — Does steeper decline travel with restrictive land use?

Descriptive only — bivariate scatters (Boulder highlighted), a composite constraint index, and a
peer-median-CCR swap. **Common-cause confound, stated up front:** the slow-growth preferences that
produced Boulder's restrictive zoning may *also* directly select for a child-free adult population;
no cross-section here can separate them.

In [ ]:
oc = []
for fips in place_proj:
    base = place_age[fips]; sa10, sa20 = school_age_total(base[2010]), school_age_total(base[2020])
    oc.append((fips, (sa20 - sa10) / sa10 * 100 if sa10 else np.nan))
h2 = pd.DataFrame(oc, columns=["place_fips", "child_change_10_20"]).merge(
    cov_df, on="place_fips").merge(places_df[["place_fips", "name", "tier"]], on="place_fips")
def z(s): return (s - s.mean()) / s.std(ddof=0)
h2["constraint_index"] = (z(h2.nza_restrictiveness) + z(np.log(h2.zhvi_2020)) +
                          z(h2.zhvi_growth_00_20) - z(h2.permits_per_1k)) / 4
print("H2 descriptive correlations with child-cohort change 2010-20:")
for col in ["nza_restrictiveness", "zhvi_2020", "permits_per_1k", "constraint_index"]:
    r, _ = stats.pearsonr(h2[col], h2["child_change_10_20"]); print(f"  vs {col:20s} r={r:+.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (col, lab) in zip(axes, [("nza_restrictiveness", "NZA restrictiveness"),
        ("zhvi_2020", "ZHVI 2020 ($)"), ("permits_per_1k", "Permits / 1k"),
        ("constraint_index", "Constraint index")]):
    isb = h2.place_fips == boulder_fips
    ax.scatter(h2.loc[~isb, col], h2.loc[~isb, "child_change_10_20"], color=CONTEXT, s=30, alpha=0.7)
    ax.scatter(h2.loc[isb, col], h2.loc[isb, "child_change_10_20"], color=HIGHLIGHT, s=90, zorder=5, label="Boulder")
    x, y = h2[col].values, h2["child_change_10_20"].values
    m, c = np.polyfit(x, y, 1); xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, m*xs + c, "k--", lw=1, alpha=0.6); ax.set_xlabel(lab); ax.set_ylabel("5-17 chg 10-20 (%)")
axes[0].legend(fontsize=9)
fig.suptitle("H2 (descriptive): child-cohort change vs land-use constraint", y=1.03)
fig.tight_layout()

In [ ]:
# peer-median-CCR swap: give Boulder peer-median youth CCR, re-project, quantify excess closed
peer_fips = peers2["place_fips"].tolist()
feeder = (AGES >= 5) & (AGES <= 27)
def recent_ccr(fips):
    base = place_age[fips]; return cohort_change_ratios(base[2010], base[2020])
peer_med = np.nanmedian(np.vstack([recent_ccr(f) for f in peer_fips]), axis=0)
adj = np.nanmean(peer_med[feeder]) / np.nanmean(recent_ccr(boulder_fips)[feeder])
sa50_own = school_age_total(place_proj[boulder_fips][2050])
sa20 = school_age_total(place_age[boulder_fips][2020])
sa50_swap = sa50_own * adj
decline = sa20 - sa50_own
print(f"peer-median-CCR swap: Boulder 2050 5-17 own {sa50_own:.0f} -> peer-typical {sa50_swap:.0f}")
if decline > 0.02 * sa20:   # only a % "share closed" is meaningful when there's a real decline
    closed = (sa50_swap - sa50_own) / decline * 100
    print(f"  share of projected decline that closes under peer-typical dynamics: {closed:+.0f}%")
else:
    closed = np.nan
    print(f"  projected decline near zero in this run ({decline:+.0f}); reporting level diff only:"
          f" {sa50_swap - sa50_own:+.0f}")

## 9 · Pinned headline numbers & analytic limits

In [ ]:
def _r(x, n=0):
    return None if x is None or not np.isfinite(x) else (round(x, n) if n else round(x))
PINNED = {
    "boulder_5_17_2020": _r(b.sa_2020), "boulder_5_17_2050": _r(b.sa_2050),
    "boulder_5_17_pct_change_2020_2050": _r(b.sa_pct_change, 1),
    "boulder_under5_pct_change": _r(b.u5_pct_change, 1),
    "steeper_decline_pctile_vs_college_towns": _r(rank_change),
    "child_share_level_pctile": _r(rank_level),
    "validation_envelope_pp": _r(ENVELOPE, 1),
    "fertility_trend_gap_5_17_2050": _r(_bff.fertility_trend_gap),
    "peer_median_swap_pct_decline_closed": _r(closed),
}
pinned_df = pd.Series(PINNED, name="value").to_frame()
pinned_df.to_csv(OUT / "pinned_numbers.csv")
h1.to_csv(OUT / "h1_metrics.csv", index=False); h2.to_csv(OUT / "h2_constraints.csv", index=False)
print("PINNED HEADLINE NUMBERS  (real only where Module 2 provenance shows the inputs live):")
print(pinned_df.to_string())
if SYNTH:
    print("\n!! place_age/permits/NZA still synthetic -> H1/H2 not yet citable; rerun nb 01 with keys.")

### Analytic limits — read before citing any number

1. **Descriptive, not causal.** H2 is association under a stated common-cause confound (slow-growth
   preferences may drive both restrictive zoning and a child-free adult population).
2. **2050 is beyond HP's validated ~15-year range**; the backtest envelope is a floor, not a CI.
3. **The control is HP-family** (Hauer ARIMA-CCR/SSP) — anchoring is consistency, not external
   validation. "Anchored to," never "confirmed against."
4. **Places don't tile counties** → damping toward county growth (λ documented), not strict control.
5. **2020 census distorts college-town 18–24 counts**, feeding the most recent CCRs for this peer
   type; GQ stripping + damping mitigate but don't erase it.
6. **County vs. city grain.** Migration evidence is strongest at county grain (IRS, DOLA); the
   headline outcome is city. The city-grain proxy is cruder by design and labeled.
7. **NZA is a current snapshot** vs. 2000–2020 outcomes and *understates* Boulder's pre-2024-reform
   restrictiveness — a directional bias on the H2 variable.
8. **Averaging/trending smooths the recent acceleration** that is the peg; show the two decades' CCRs
   side by side in prose rather than averaging it away.
9. **The HP youth CCR already contains net migration**, so projection and decomposition are not
   independent confirmations.
10. **Annexation contaminates aggressive-annexer peers'** CCRs; NHGIS crosswalks mitigate imperfectly.
    Boulder's hard growth boundary makes it unusually clean.

*Provenance governs citeability: per the Module 2 manifest, any input still `synthetic` (here:
place single-year age, permits, NZA pending an IPUMS key + the Hauer OSF pull) makes the dependent
H1/H2 numbers non-citable until notebook 01 is rerun with those sources live.*